In [1]:
!pip install "zarr>=3" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.7 MB/s eta 0:00:00:00:010:01


In [2]:
import json
import os

import blosc2
import numpy as np
import pandas as pd
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy.ndimage import uniform_filter
from scipy.optimize import linear_sum_assignment

TRAIN_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X um/voxel
DOWNSAMPLE_Z = 1
DOWNSAMPLE_XY = 4
MIN_PEAK_DISTANCE = 3  # voxels, in downsampled-XY space - tune this later
MAX_LINK_DISTANCE = 15.0  # um

dataset_name = '44b6_0113de3b'
zarr_path = os.path.join(TRAIN_DIR, dataset_name + '.zarr')

with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
    arr_meta = json.load(f)
shape = tuple(arr_meta['shape'])  # (T, Z, Y, X)
dtype = np.dtype(arr_meta['data_type'])
n_t = shape[0]

prev_centroids = {}
node_id_counter = 1
all_rows = []

for t in range(n_t):
    chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    with open(chunk_path, 'rb') as f:
        compressed = f.read()
    decompressed = blosc2.decompress(compressed)
    vol = np.frombuffer(decompressed, dtype=dtype).reshape(shape[1:])  # (Z, Y, X)

    ds = vol[::DOWNSAMPLE_Z, ::DOWNSAMPLE_XY, ::DOWNSAMPLE_XY]
    smoothed = uniform_filter(ds.astype(np.float32), size=3)

    threshold = threshold_otsu(smoothed)
    binary = smoothed > threshold

    # --- watershed splitting of touching/adjacent nuclei ---
    distance = ndi.distance_transform_edt(binary)
    coords = peak_local_max(distance, min_distance=MIN_PEAK_DISTANCE, labels=binary)
    peak_mask = np.zeros_like(distance, dtype=bool)
    if len(coords) > 0:
        peak_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(peak_mask)
    labeled = watershed(-distance, markers, mask=binary)
    n_features = labeled.max()
    # --------------------------------------------------------

    centroids = {}
    for comp_id in range(1, n_features + 1):
        coords_comp = np.argwhere(labeled == comp_id)
        if len(coords_comp) == 0:
            continue
        centroid = coords_comp.mean(axis=0) * np.array([DOWNSAMPLE_Z, DOWNSAMPLE_XY, DOWNSAMPLE_XY])
        nid = node_id_counter
        node_id_counter += 1
        centroids[nid] = (int(round(centroid[0])), int(round(centroid[1])), int(round(centroid[2])))
        all_rows.append({
            'dataset': dataset_name,
            'row_type': 'node',
            'node_id': nid,
            't': t,
            'z': centroids[nid][0],
            'y': centroids[nid][1],
            'x': centroids[nid][2],
            'source_id': -1,
            'target_id': -1,
        })

    if prev_centroids:
        prev_ids = list(prev_centroids.keys())
        curr_ids = list(centroids.keys())
        if prev_ids and curr_ids:
            prev_coords = np.array([prev_centroids[pid] for pid in prev_ids], dtype=np.float64)
            curr_coords = np.array([centroids[cid] for cid in curr_ids], dtype=np.float64)
            prev_phys = prev_coords * SCALE
            curr_phys = curr_coords * SCALE
            dist = np.sqrt(((prev_phys[:, None] - curr_phys[None, :]) ** 2).sum(axis=2))
            row_ind, col_ind = linear_sum_assignment(dist)
            for ri, ci in zip(row_ind, col_ind):
                if dist[ri, ci] <= MAX_LINK_DISTANCE:
                    all_rows.append({
                        'dataset': dataset_name,
                        'row_type': 'edge',
                        'node_id': -1,
                        't': -1,
                        'z': -1,
                        'y': -1,
                        'x': -1,
                        'source_id': prev_ids[ri],
                        'target_id': curr_ids[ci],
                    })

    prev_centroids = centroids

print(f'{dataset_name}: {node_id_counter - 1} nodes')

submission = pd.DataFrame(all_rows)
submission.index.name = 'id'
submission.to_csv('submission_train_sample.csv')
print(f'Done. {len(submission)} rows written to submission_train_sample.csv')

44b6_0113de3b: 15181 nodes
Done. 29075 rows written to submission_train_sample.csv


In [3]:
import os
import numpy as np
import pandas as pd
import zarr
from scipy.optimize import linear_sum_assignment

SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X um/voxel
MAX_MATCH_DIST = 7.0  # um
OVERPREDICTION_WEIGHT = 0.1  # 'a' in the adjusted-jaccard formula


def load_geff_ground_truth(geff_path):
    root = zarr.open(geff_path, mode="r")
    node_ids = np.asarray(root["nodes/ids"][:])
    t = np.asarray(root["nodes/props/t/values"][:])
    z = np.asarray(root["nodes/props/z/values"][:])
    y = np.asarray(root["nodes/props/y/values"][:])
    x = np.asarray(root["nodes/props/x/values"][:])
    edges = np.asarray(root["edges/ids"][:])
    nodes_df = pd.DataFrame({"node_id": node_ids, "t": t, "z": z, "y": y, "x": x})
    edges_df = pd.DataFrame(edges, columns=["source_id", "target_id"])
    t_true = root.attrs["geff"]["extra"]["estimated_number_of_nodes"]
    return nodes_df, edges_df, t_true


def load_submission_for_dataset(submission_df, dataset_name):
    sub = submission_df[submission_df["dataset"] == dataset_name]
    pred_nodes = sub[sub["row_type"] == "node"][["node_id", "t", "z", "y", "x"]].copy()
    pred_edges = sub[sub["row_type"] == "edge"][["source_id", "target_id"]].copy()
    return pred_nodes, pred_edges


def match_nodes_per_timepoint(gt_nodes, pred_nodes, max_dist=MAX_MATCH_DIST):
    match_map = {}
    for t in sorted(gt_nodes["t"].unique()):
        gt_t = gt_nodes[gt_nodes["t"] == t]
        pred_t = pred_nodes[pred_nodes["t"] == t]
        if len(gt_t) == 0 or len(pred_t) == 0:
            continue
        gt_coords = gt_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * SCALE
        pred_coords = pred_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * SCALE
        dist = np.sqrt(((pred_coords[:, None, :] - gt_coords[None, :, :]) ** 2).sum(axis=2))
        row_ind, col_ind = linear_sum_assignment(dist)
        for ri, ci in zip(row_ind, col_ind):
            if dist[ri, ci] <= max_dist:
                match_map[int(pred_t.iloc[ri]["node_id"])] = int(gt_t.iloc[ci]["node_id"])
    return match_map


def edge_jaccard(gt_edges, pred_edges, match_map):
    """Implements the exact metrics.md edge-matching rule:
    - unmatched-endpoint edges are IGNORED (not FP)
    - FP only when a matched-node edge conflicts with that node's REAL gt connection
    """
    gt_edge_set = set(map(tuple, gt_edges[["source_id", "target_id"]].to_numpy()))
    gt_sources = set(gt_edges["source_id"])
    gt_targets = set(gt_edges["target_id"])

    tp, fp = 0, 0
    matched_gt_edges = set()

    for _, row in pred_edges.iterrows():
        src_gt = match_map.get(int(row["source_id"]))
        tgt_gt = match_map.get(int(row["target_id"]))
        if src_gt is None or tgt_gt is None:
            continue  # ignored - unmatched endpoint

        if (src_gt, tgt_gt) in gt_edge_set:
            tp += 1
            matched_gt_edges.add((src_gt, tgt_gt))
            continue

        # not a TP - check the two specific FP conditions
        cond_a = tgt_gt in gt_targets  # tgt_gt really connects to some OTHER source
        cond_b = src_gt in gt_sources  # src_gt really connects to some OTHER target
        if cond_a or cond_b:
            fp += 1
        # else: ignored (matched nodes, but neither has any real GT connection at all)

    fn = len(gt_edge_set) - len(matched_gt_edges)
    return tp, fp, fn


def score_dataset(train_dir, submission_df, dataset_name):
    geff_path = os.path.join(train_dir, dataset_name + ".geff")
    gt_nodes, gt_edges, t_true = load_geff_ground_truth(geff_path)
    pred_nodes, pred_edges = load_submission_for_dataset(submission_df, dataset_name)

    match_map = match_nodes_per_timepoint(gt_nodes, pred_nodes)
    tp, fp, fn = edge_jaccard(gt_edges, pred_edges, match_map)

    raw_jaccard = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    t_pred = len(pred_nodes)
    penalty_term = 1 - OVERPREDICTION_WEIGHT * (t_pred - t_true) / t_true
    adjusted_jaccard = max(0.0, raw_jaccard * penalty_term)

    return {
        "dataset": dataset_name,
        "tp": tp, "fp": fp, "fn": fn,
        "n_pred_nodes": t_pred, "t_true_estimate": t_true,
        "raw_edge_jaccard": raw_jaccard,
        "adjusted_edge_jaccard": adjusted_jaccard,
    }


# --- run it ---
train_dir = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
submission_df = pd.read_csv('submission_train_sample.csv')
dataset_name = '44b6_0113de3b'

result = score_dataset(train_dir, submission_df, dataset_name)
for k, v in result.items():
    print(f"{k}: {v}")

dataset: 44b6_0113de3b
tp: 28
fp: 0
fn: 22
n_pred_nodes: 15181
t_true_estimate: 25755
raw_edge_jaccard: 0.56
adjusted_edge_jaccard: 0.5829914191419143


In [4]:
Z_TOLERANCE = 3  # voxels, roughly one nucleus-thickness in z

pred_t = submission_df[
    (submission_df['dataset'] == dataset_name) &
    (submission_df['row_type'] == 'node') &
    (submission_df['t'] == t0) &
    (submission_df['z'] >= z0 - Z_TOLERANCE) &
    (submission_df['z'] <= z0 + Z_TOLERANCE)
]

plt.figure(figsize=(8, 8))
plt.imshow(vol[z0], cmap='gray', vmax=np.percentile(vol[z0], 99.5))
plt.scatter(pred_t['x'], pred_t['y'], s=8, c='cyan', label=f'predicted (z within ±{Z_TOLERANCE})')
plt.scatter([x0], [y0], s=150, c='red', marker='x', linewidths=3, label='ground truth')
plt.legend()
plt.title(f't={t0}, z-slice={z0}')
plt.show()

NameError: name 't0' is not defined

In [5]:
print(gt_nodes['t'].value_counts().sort_index())

NameError: name 'gt_nodes' is not defined

In [ ]:
import zarr
geff_root = zarr.open(os.path.join(train_dir, dataset_name + '.geff'), mode='r')
print(geff_root.attrs.get('estimated_number_of_nodes'))

In [ ]:
import zarr

geff_root = zarr.open(os.path.join(train_dir, dataset_name + '.geff'), mode='r')

print("Root attrs:", dict(geff_root.attrs))
print()
print("Tree structure:")
print(geff_root.tree())
print()

# check attrs on every subgroup/array, in case it's nested
def walk_attrs(group, prefix=""):
    for key in group.keys():
        item = group[key]
        attrs = dict(item.attrs)
        if attrs:
            print(f"{prefix}{key} attrs: {attrs}")
        if hasattr(item, 'keys'):  # it's a group, recurse
            walk_attrs(item, prefix=f"{prefix}{key}/")

walk_attrs(geff_root)

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/royerlab/kaggle-cell-tracking-competition/main/metrics.md"
with urllib.request.urlopen(url) as response:
    metrics_content = response.read().decode('utf-8')
print(metrics_content)

In [ ]:
import json
import os

import blosc2
import numpy as np
import pandas as pd
import zarr
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy.ndimage import uniform_filter
from scipy.optimize import linear_sum_assignment

TRAIN_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = np.array([1.625, 0.40625, 0.40625])
DOWNSAMPLE_Z = 1
DOWNSAMPLE_XY = 4
MIN_PEAK_DISTANCE = 3
MAX_LINK_DISTANCE = 15.0
MAX_MATCH_DIST = 7.0
OVERPREDICTION_WEIGHT = 0.1

# --- embryo count check ---
train_zarr_folders = sorted(
    d.replace('.zarr', '') for d in os.listdir(TRAIN_DIR) if d.endswith('.zarr')
)
embryo_ids = sorted(set(name.split('_')[0] for name in train_zarr_folders))
print(f"Total train samples: {len(train_zarr_folders)}")
print(f"Unique embryo IDs: {len(embryo_ids)} -> {embryo_ids}")
print()


def detect_and_link(dataset_name, base_dir):
    zarr_path = os.path.join(base_dir, dataset_name + '.zarr')
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        arr_meta = json.load(f)
    shape = tuple(arr_meta['shape'])
    dtype = np.dtype(arr_meta['data_type'])
    n_t = shape[0]

    prev_centroids = {}
    node_id_counter = 1
    rows = []

    for t in range(n_t):
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as f:
            vol = np.frombuffer(blosc2.decompress(f.read()), dtype=dtype).reshape(shape[1:])

        ds = vol[::DOWNSAMPLE_Z, ::DOWNSAMPLE_XY, ::DOWNSAMPLE_XY]
        smoothed = uniform_filter(ds.astype(np.float32), size=3)
        threshold = threshold_otsu(smoothed)
        binary = smoothed > threshold

        distance = ndi.distance_transform_edt(binary)
        coords = peak_local_max(distance, min_distance=MIN_PEAK_DISTANCE, labels=binary)
        peak_mask = np.zeros_like(distance, dtype=bool)
        if len(coords) > 0:
            peak_mask[tuple(coords.T)] = True
        markers, _ = ndi.label(peak_mask)
        labeled = watershed(-distance, markers, mask=binary)
        n_features = labeled.max()

        centroids = {}
        for comp_id in range(1, n_features + 1):
            coords_comp = np.argwhere(labeled == comp_id)
            if len(coords_comp) == 0:
                continue
            centroid = coords_comp.mean(axis=0) * np.array([DOWNSAMPLE_Z, DOWNSAMPLE_XY, DOWNSAMPLE_XY])
            nid = node_id_counter
            node_id_counter += 1
            centroids[nid] = tuple(int(round(c)) for c in centroid)
            rows.append({
                'dataset': dataset_name, 'row_type': 'node', 'node_id': nid, 't': t,
                'z': centroids[nid][0], 'y': centroids[nid][1], 'x': centroids[nid][2],
                'source_id': -1, 'target_id': -1,
            })

        if prev_centroids and centroids:
            prev_ids, curr_ids = list(prev_centroids.keys()), list(centroids.keys())
            prev_phys = np.array([prev_centroids[p] for p in prev_ids], dtype=np.float64) * SCALE
            curr_phys = np.array([centroids[c] for c in curr_ids], dtype=np.float64) * SCALE
            dist = np.sqrt(((prev_phys[:, None] - curr_phys[None, :]) ** 2).sum(axis=2))
            row_ind, col_ind = linear_sum_assignment(dist)
            for ri, ci in zip(row_ind, col_ind):
                if dist[ri, ci] <= MAX_LINK_DISTANCE:
                    rows.append({
                        'dataset': dataset_name, 'row_type': 'edge', 'node_id': -1, 't': -1,
                        'z': -1, 'y': -1, 'x': -1,
                        'source_id': prev_ids[ri], 'target_id': curr_ids[ci],
                    })
        prev_centroids = centroids

    return rows


def load_geff_ground_truth(geff_path):
    root = zarr.open(geff_path, mode="r")
    node_ids = np.asarray(root["nodes/ids"][:])
    t = np.asarray(root["nodes/props/t/values"][:])
    z = np.asarray(root["nodes/props/z/values"][:])
    y = np.asarray(root["nodes/props/y/values"][:])
    x = np.asarray(root["nodes/props/x/values"][:])
    edges = np.asarray(root["edges/ids"][:])
    nodes_df = pd.DataFrame({"node_id": node_ids, "t": t, "z": z, "y": y, "x": x})
    edges_df = pd.DataFrame(edges, columns=["source_id", "target_id"])
    t_true = root.attrs["geff"]["extra"]["estimated_number_of_nodes"]
    return nodes_df, edges_df, t_true


def match_nodes_per_timepoint(gt_nodes, pred_nodes, max_dist=MAX_MATCH_DIST):
    match_map = {}
    for t in sorted(gt_nodes["t"].unique()):
        gt_t = gt_nodes[gt_nodes["t"] == t]
        pred_t = pred_nodes[pred_nodes["t"] == t]
        if len(gt_t) == 0 or len(pred_t) == 0:
            continue
        gt_coords = gt_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * SCALE
        pred_coords = pred_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * SCALE
        dist = np.sqrt(((pred_coords[:, None, :] - gt_coords[None, :, :]) ** 2).sum(axis=2))
        row_ind, col_ind = linear_sum_assignment(dist)
        for ri, ci in zip(row_ind, col_ind):
            if dist[ri, ci] <= max_dist:
                match_map[int(pred_t.iloc[ri]["node_id"])] = int(gt_t.iloc[ci]["node_id"])
    return match_map


def edge_jaccard_counts(gt_edges, pred_edges, match_map):
    gt_edge_set = set(map(tuple, gt_edges[["source_id", "target_id"]].to_numpy()))
    gt_sources = set(gt_edges["source_id"])
    gt_targets = set(gt_edges["target_id"])
    tp, fp = 0, 0
    matched_gt_edges = set()
    for _, row in pred_edges.iterrows():
        src_gt = match_map.get(int(row["source_id"]))
        tgt_gt = match_map.get(int(row["target_id"]))
        if src_gt is None or tgt_gt is None:
            continue
        if (src_gt, tgt_gt) in gt_edge_set:
            tp += 1
            matched_gt_edges.add((src_gt, tgt_gt))
            continue
        if tgt_gt in gt_targets or src_gt in gt_sources:
            fp += 1
    fn = len(gt_edge_set) - len(matched_gt_edges)
    return tp, fp, fn


def score_dataset(train_dir, submission_df, dataset_name):
    geff_path = os.path.join(train_dir, dataset_name + ".geff")
    gt_nodes, gt_edges, t_true = load_geff_ground_truth(geff_path)
    sub = submission_df[submission_df["dataset"] == dataset_name]
    pred_nodes = sub[sub["row_type"] == "node"][["node_id", "t", "z", "y", "x"]]
    pred_edges = sub[sub["row_type"] == "edge"][["source_id", "target_id"]]

    match_map = match_nodes_per_timepoint(gt_nodes, pred_nodes)
    tp, fp, fn = edge_jaccard_counts(gt_edges, pred_edges, match_map)

    raw_jaccard = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    t_pred = len(pred_nodes)
    penalty_term = 1 - OVERPREDICTION_WEIGHT * (t_pred - t_true) / t_true
    adjusted_jaccard = max(0.0, raw_jaccard * penalty_term)

    return {
        "dataset": dataset_name, "tp": tp, "fp": fp, "fn": fn,
        "w": tp + fp + fn, "adjusted_edge_jaccard": adjusted_jaccard,
    }


# --- run detection+linking across ALL train samples, then score all ---
all_rows = []
per_sample_results = []

for dataset_name in train_zarr_folders:
    print(f"Processing {dataset_name}...")
    rows = detect_and_link(dataset_name, TRAIN_DIR)
    all_rows.extend(rows)

submission_all = pd.DataFrame(all_rows)
submission_all.index.name = 'id'
submission_all.to_csv('submission_all_train.csv')

for dataset_name in train_zarr_folders:
    result = score_dataset(TRAIN_DIR, submission_all, dataset_name)
    per_sample_results.append(result)
    print(result)

# --- proper weighted aggregation per metrics.md ---
results_df = pd.DataFrame(per_sample_results)
total_w = results_df['w'].sum()
weighted_adjusted_jaccard = (results_df['adjusted_edge_jaccard'] * results_df['w']).sum() / total_w

print()
print(f"=== AGGREGATE (weight-averaged by TP+FP+FN, per metrics.md) ===")
print(f"Overall adjusted_edge_jaccard: {weighted_adjusted_jaccard:.4f}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import zarr
import blosc2
import matplotlib.pyplot as plt

train_dir = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
dataset_name = '44b6_53f95252'

# --- load GT ---
gt_root = zarr.open(os.path.join(train_dir, dataset_name + '.geff'), mode='r')
gt_nodes = pd.DataFrame({
    'node_id': np.asarray(gt_root['nodes/ids'][:]),
    't': np.asarray(gt_root['nodes/props/t/values'][:]),
    'z': np.asarray(gt_root['nodes/props/z/values'][:]),
    'y': np.asarray(gt_root['nodes/props/y/values'][:]),
    'x': np.asarray(gt_root['nodes/props/x/values'][:]),
})
print(f"GT nodes for {dataset_name}: {len(gt_nodes)}")
print(gt_nodes[['t', 'z', 'y', 'x']])

# --- pick the first GT node and load its raw frame ---
example = gt_nodes.iloc[0]
t0, z0, y0, x0 = int(example.t), int(example.z), int(example.y), int(example.x)
print(f"\nGT node at t={t0}, z={z0}, y={y0}, x={x0}")

zarr_path = os.path.join(train_dir, dataset_name + '.zarr')
with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
    arr_meta = json.load(f)
shape = tuple(arr_meta['shape'])
dtype = np.dtype(arr_meta['data_type'])
chunk_path = os.path.join(zarr_path, '0', 'c', str(t0), '0', '0', '0')
with open(chunk_path, 'rb') as f:
    vol = np.frombuffer(blosc2.decompress(f.read()), dtype=dtype).reshape(shape[1:])

# --- load your predictions for this sample/timepoint (from the full-train run) ---
submission_df = pd.read_csv('submission_all_train.csv')
Z_TOLERANCE = 3
pred_t = submission_df[
    (submission_df['dataset'] == dataset_name) &
    (submission_df['row_type'] == 'node') &
    (submission_df['t'] == t0) &
    (submission_df['z'] >= z0 - Z_TOLERANCE) &
    (submission_df['z'] <= z0 + Z_TOLERANCE)
]
print(f"\nYour predicted nodes near this z-slice: {len(pred_t)}")

plt.figure(figsize=(8, 8))
plt.imshow(vol[z0], cmap='gray', vmax=np.percentile(vol[z0], 99.5))
plt.scatter(pred_t['x'], pred_t['y'], s=8, c='cyan', label=f'predicted (z within +/-{Z_TOLERANCE})')
plt.scatter([x0], [y0], s=150, c='red', marker='x', linewidths=3, label='ground truth')
plt.legend()
plt.title(f'{dataset_name}: t={t0}, z-slice={z0}')
plt.show()

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import blosc2
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt

train_dir = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
dataset_name = '44b6_53f95252'
SCALE = np.array([1.625, 0.40625, 0.40625])

t0, z0 = 0, 49  # the same GT node we just looked at

zarr_path = os.path.join(train_dir, dataset_name + '.zarr')
with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
    arr_meta = json.load(f)
shape = tuple(arr_meta['shape'])
dtype = np.dtype(arr_meta['data_type'])
chunk_path = os.path.join(zarr_path, '0', 'c', str(t0), '0', '0', '0')
with open(chunk_path, 'rb') as f:
    vol = np.frombuffer(blosc2.decompress(f.read()), dtype=dtype).reshape(shape[1:])

# try a few downsample factors on this single frame, no linking, just detection
for downsample_xy in [4, 2, 1]:
    ds = vol[::1, ::downsample_xy, ::downsample_xy]  # keep Z at full res, vary XY only
    smoothed = uniform_filter(ds.astype(np.float32), size=3)
    threshold = threshold_otsu(smoothed)
    binary = smoothed > threshold

    distance = ndi.distance_transform_edt(binary)
    coords = peak_local_max(distance, min_distance=3, labels=binary)
    peak_mask = np.zeros_like(distance, dtype=bool)
    if len(coords) > 0:
        peak_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(peak_mask)
    labeled = watershed(-distance, markers, mask=binary)
    n_features = labeled.max()

    centroids_xy = []
    for comp_id in range(1, n_features + 1):
        coords_comp = np.argwhere(labeled == comp_id)
        if len(coords_comp) == 0:
            continue
        c = coords_comp.mean(axis=0) * np.array([1, downsample_xy, downsample_xy])
        centroids_xy.append((c[1], c[2]))  # (y, x) in full-res voxels, this z-slab only

    print(f"downsample_xy={downsample_xy}: {n_features} components found (this frame, all z)")

    # plot just the ones near z0
    plt.figure(figsize=(6, 6))
    plt.imshow(vol[z0], cmap='gray', vmax=np.percentile(vol[z0], 99.5))
    if centroids_xy:
        ys, xs = zip(*centroids_xy)
        plt.scatter(xs, ys, s=8, c='cyan')
    plt.scatter([173], [120], s=150, c='red', marker='x', linewidths=3)  # the GT point
    plt.title(f'downsample_xy={downsample_xy}, {n_features} total components')
    plt.show()

In [ ]:
import os
import json
import time

import blosc2
import numpy as np
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy.ndimage import uniform_filter

train_dir = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
dataset_name = '44b6_53f95252'  # the sample we've been diagnosing

zarr_path = os.path.join(train_dir, dataset_name + '.zarr')
with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
    arr_meta = json.load(f)
shape = tuple(arr_meta['shape'])
dtype = np.dtype(arr_meta['data_type'])
n_t = shape[0]

for downsample_xy in [4, 2, 1]:
    start = time.time()
    total_components = 0

    for t in range(n_t):
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as f:
            vol = np.frombuffer(blosc2.decompress(f.read()), dtype=dtype).reshape(shape[1:])

        ds = vol[::1, ::downsample_xy, ::downsample_xy]
        smoothed = uniform_filter(ds.astype(np.float32), size=3)
        threshold = threshold_otsu(smoothed)
        binary = smoothed > threshold

        distance = ndi.distance_transform_edt(binary)
        coords = peak_local_max(distance, min_distance=3, labels=binary)
        peak_mask = np.zeros_like(distance, dtype=bool)
        if len(coords) > 0:
            peak_mask[tuple(coords.T)] = True
        markers, _ = ndi.label(peak_mask)
        labeled = watershed(-distance, markers, mask=binary)
        total_components += labeled.max()

    elapsed = time.time() - start
    print(f"downsample_xy={downsample_xy}: {elapsed:.1f}s for {n_t} timepoints "
          f"({elapsed/n_t:.3f}s/frame), {total_components} total components across video")
    print(f"  -> extrapolated to 199 samples: {elapsed*199/60:.1f} minutes")
    print(f"  -> extrapolated to a ~275-sample hidden test set (train-sized estimate): {elapsed*275/60:.1f} minutes")
    print()